In [ ]:
sim = 'sim2419'

# settings
cell_type='ispn'
model = 3

import os, sys
# compute absolute path of main folder
cwd = os.getcwd()

if os.path.exists(os.path.join(cwd, 'settings.py')):
    main_dir = cwd
else:
    main_dir = os.path.abspath(os.path.join(cwd, '..'))

# set CWD or add to path
os.chdir(main_dir)
sys.path.insert(0, main_dir)

# then import/run settings
%run -i settings.py

from analysis_functions import *

# Get the current working directory
current_wd = os.getcwd()
# 'sim' + str(sim)

downsample = True

home = os.path.expanduser('~')

# Set working directory
external = False
external_name = 'MacOS10'
target = f'model{model}'

if not external:
    if downsample:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = home + '/Documents/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'
else:
    if downsample:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/downsample/' + cell_type + '/' + target + '/physiological/simulations'
    else:
        wd = '/Volumes/' + external_name + '/Repositories/msNEURON_Belal2026/' + cell_type + '/' + target + '/physiological/simulations'


# create path to directory to save analysis
base_path = os.path.join(home, 'Documents', 'Repositories', 'msNEURON_Belal2026', 'analysis')
sim_image_path = os.path.join(base_path, cell_type, sim)
os.makedirs(sim_image_path, exist_ok=True)

save = False

In [ ]:
# Basic plot settings
s = 5 # splprep fits a parametric B-spline to your 2D points 
lwd = 1 # standardise line width

height1 = 600
width1 = 600

height2 = 600
width2 = 800

if cell_type == 'ispn':
    plane='yx'
    mirror=False
    dend_name = 'dend[12]'
    
else:
    plane='xy'
    mirror=False
    dend_name = 'dend[7]'

In [ ]:
# load simulations
sims_dir = os.path.join(wd, sim)
# sim_data = load_data_dicts(sims_dir, verbose=True)

sim_data = load_data_dicts(wd=wd, sim=sim, cell_type=cell_type, verbose=True)

In [ ]:
# check files loaded in correct order
for name in sim_data.keys():
    print(name)

In [ ]:
report_memory(verbose=True)

In [ ]:
# check if coordinates exist
coords_exist = summarise_cell_coordinates(sim_data)

# if so load from v_all_3D else retrieve from cell_build
# this is because the methods to retrieve the cell_coordinates differ so it is important when plotting 3D heatmaps that the correct coordinates are used
# the method to default will only be used when the simulations have no 3d data 
if coords_exist:
    first_sim = next(iter(sim_data.values()))
    v_all_3D = first_sim['v_all_3D']
    first_key = next(iter(v_all_3D)) 
    
    cell_coordinates = np.array(v_all_3D[first_key]['cell_coordinates'], dtype=object)
    _, _, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=None)
else:
    cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=None)
    cell_coordinates = []
    
    for sec in cell.somalist:
        h('access ' + sec.name())
        x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
        cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])
    
    # Setup for dendritic sections
    for sec in cell.dendlist:
        for seg in sec:
            x, y, z, diam = interpolate_3d(sec, seg.x)
            cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])
        
    cell_coordinates = np.array(cell_coordinates, dtype=object)

    
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=lwd, s=s, color='grey', height=height1, width=width1, plane=plane, mirror=mirror)

fig_morphology.show(config={"toImageButtonOptions": {"format": "svg"}})
if save:
    fig_morphology.write_image(f"{sim_image_path}/morphology.svg", format='svg', scale=3)


In [ ]:
# Morphology plot with synaptic locations

# extract names and locations of GLUT and GABA inputs from metadata
# using last simulation to get all GABA locations used (fixed)
dend_gaba_all = list(sim_data.values())[-1]['metadata']['dend_gaba']
gaba_locations_all = list(sim_data.values())[-1]['metadata']['gaba_locations']

dend_glut = list(sim_data.values())[-1]['metadata']['dend_glut']
glut_locations = list(sim_data.values())[-1]['metadata']['glutamate_locations']

if len(dend_glut) != len(glut_locations): # one upstate site 
    dend_glut = dend_glut * len(glut_locations)
    
# indices
idxs_gaba = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite=dend_gaba_all, target_location=gaba_locations_all)
idxs_glut = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite=dend_glut, target_location=glut_locations)

# coordinates
coords_gaba = get_coord_index_interp(cell_coordinates=cell_coordinates, target_dendrite=dend_gaba_all, target_location=gaba_locations_all)
coords_glut = get_coord_index_interp(cell_coordinates=cell_coordinates, target_dendrite=dend_glut, target_location=glut_locations)

# with interpolation
fig_morphology = morphology_plot(cell_coordinates=cell_coordinates, dend_tree=dend_tree, idxs=[coords_gaba, coords_glut], 
                                 idxs_colors = ['#5393CF', '#CD5C5C'], dot_size=[5,6], lwd=lwd, s=s, color='grey', 
                                 height=height1, width=width1, plane=plane, mirror=mirror, alpha=[0.5, 0.25])

fig_path_morphology = morphology_plot(cell_coordinates=cell_coordinates, dend_tree=dend_tree, dend_name=dend_name, idxs=[coords_gaba, coords_glut], 
                                 idxs_colors = ['#5393CF', '#CD5C5C'], dot_size=[5,6], lwd=lwd, s=s, color='grey', 
                                 height=height1, width=width1, plane=plane, mirror=mirror, alpha=[0.5, 0.25])


fig_morphology.show(config={"toImageButtonOptions": {"format": "svg"}})

fig_path_morphology.show(config={"toImageButtonOptions": {"format": "svg"}})


if save:
    fig_morphology.write_image(f"{sim_image_path}/morphology2.svg", format='svg', scale=3)

In [ ]:
compare_synapse_coords("Glutamatergic", dend_glut, glut_locations, coords_glut)

In [ ]:
compare_synapse_coords("GABAergic", dend_gaba_all, gaba_locations_all, coords_gaba)

In [ ]:
# Simple output traces
sim_time = sim_data['a']['metadata']['sim_time']

dt = next(iter(sim_data.values()))['metadata']['dt']

if downsample:
    ds_plot = 1
    dt = ds * dt
    
else:
    ds_plot = 10
    
Vsoma = []
Vdend = []

for variables in sim_data.values():
    if 'vsoma' in variables:
        Vsoma.append(variables['vsoma'])
    if 'vdend' in variables:
        Vdend.append(variables['vdend'])    

Vsoma_fig = plot3_dt(Vsoma, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='somatic voltage', 
                yrange=[-86, 40], yabline = [-86, -60], black_trace=None, x_err_bar=50, baseline=50, 
                     dt=dt, ds=ds_plot, offset=True, offset_ms=150, offset_y=None)        

Vdend_fig = plot3_dt(Vdend, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='dendritic voltage', 
                yrange=[-86, -10], yabline = [-86, -20], black_trace=None, x_err_bar=50, baseline=50, 
                     dt=dt, ds=ds_plot, offset=True, offset_ms=150, offset_y=None)    

Vsoma_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
Vdend_fig.show(config={"toImageButtonOptions": {"format": "svg"}})

if save:
    Vsoma_fig.write_image(f"{sim_image_path}/Vsoma.svg", format='svg', scale=3)
    Vdend_fig.write_image(f"{sim_image_path}/Vdend.svg", format='svg', scale=3)
    

In [ ]:
record_dendrite = next(iter(sim_data.values()))['metadata']['record_dendrite']
record_location = next(iter(sim_data.values()))['metadata']['record_location']

# get indices for the recording location
record_coords = get_coord_index_interp(cell_coordinates=cell_coordinates, target_dendrite=record_dendrite, target_location=record_location)
idx = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite=record_dendrite, target_location=record_location)

compare_synapse_coords("Glutamatergic", record_dendrite, record_location, record_coords)

compare_synapse_coords("Glutamatergic", record_dendrite, record_location, cell_coordinates[idx][0:5,])


In [ ]:
# Simple output traces can also be obtained from 'v_all_3D'
idx1 = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite='soma[0]', target_location=0.5)
idx2 = get_coord_index(cell_coordinates=cell_coordinates, target_dendrite=record_dendrite, target_location=record_location)

Vsoma1 = []
Vdend1 = []

for sim_vars in sim_data.values():
    v_all = sim_vars['v_all_3D']
    for sim_block in v_all.values():           
        v_arrays = sim_block['v']              
        Vsoma1.append(v_arrays[idx1])
        Vdend1.append(v_arrays[idx2])

In [ ]:
Vsoma1_fig = plot3_dt(Vsoma1, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='somatic voltage', 
                yrange=[-86, 40], yabline = [-86, -60], black_trace=None, x_err_bar=50, baseline=50, 
                     dt=dt, ds=ds_plot, offset=True, offset_ms=150, offset_y=None)        

Vdend1_fig = plot3_dt(Vdend1, yaxis='V (mV)', stim_time = 150, sim_time=sim_time, title='dendritic voltage', 
                yrange=[-86, -10], yabline = [-86, -20], black_trace=None, x_err_bar=50, baseline=50, 
                     dt=dt, ds=ds_plot, offset=True, offset_ms=150, offset_y=None)      

Vsoma1_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
Vdend1_fig.show(config={"toImageButtonOptions": {"format": "svg"}})


In [ ]:
# 3D heat map
Vpeaks_3D  = []

for sim_vars in sim_data.values():
    v_all = sim_vars['v_all_3D']
    max_vals = []
    for sim_block in v_all.values():
        v_arrays = sim_block['v']                 # shape: (n_traces, n_timepoints)
        # get maximum of each trace
        max_vals.extend([np.max(trace) for trace in v_arrays])
    Vpeaks_3D.append(np.array(max_vals))
        

In [ ]:
# be careful with too much downsampling; it can really alter the image

# num_gabas = 4
idx = list(sim_data.keys()).index(next(k for k, d in sim_data.items() if d['metadata']['num_gabas'] == 4))

V2D_fig  = heatmap2D(cell_coordinates=cell_coordinates, dend_tree=dend_tree, z=Vpeaks_3D[idx], palette='oleron', reverse=False, alpha=1, 
                     lwd=lwd, show_bar=True, title='', zmin=-85, zmax=40, height=height1, width=width1, scale_bar=50, 
                     x_range=[-125, 175], y_range=[-150, 150], plane=plane, mirror=mirror, s=s, ds=2)

V2D_path_fig  = heatmap2D(cell_coordinates=cell_coordinates, dend_tree=dend_tree, dend_name=dend_name, z=Vpeaks_3D[idx], palette='oleron', reverse=False, alpha=1, 
                     lwd=lwd, show_bar=True, title='', zmin=-85, zmax=40, height=height1, width=width1, scale_bar=50, 
                     x_range=[-125, 175], y_range=[-150, 150], plane=plane, mirror=False, s=s, ds=2)


V2D_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
V2D_path_fig.show(config={"toImageButtonOptions": {"format": "svg"}})


if save:
    V2D_fig.write_image(f"{sim_image_path}/V2D_{idx}.svg", format='svg', scale=3)
    V2D_path_fig.write_image(f"{sim_image_path}/V2D_path_{idx}.svg", format='svg', scale=3)

    

In [ ]:
# num_gabas = 40
idx = list(sim_data.keys()).index(next(k for k, d in sim_data.items() if d['metadata']['num_gabas'] == 40))

V2D_fig  = heatmap2D(cell_coordinates=cell_coordinates, dend_tree=dend_tree, z=Vpeaks_3D[idx], palette='oleron', reverse=False, alpha=1, 
                     lwd=lwd, show_bar=True, title='', zmin=-85, zmax=40, height=height1, width=width1, scale_bar=50, 
                     x_range=[-125, 175], y_range=[-150, 150], plane=plane, mirror=mirror, s=s, ds=2)

V2D_path_fig  = heatmap2D(cell_coordinates=cell_coordinates, dend_tree=dend_tree, dend_name=dend_name, z=Vpeaks_3D[idx], palette='oleron', reverse=False, alpha=1, 
                     lwd=lwd, show_bar=True, title='', zmin=-85, zmax=40, height=height1, width=width1, scale_bar=50, 
                     x_range=[-125, 175], y_range=[-150, 150], plane=plane, mirror=False, s=s, ds=2)


V2D_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
V2D_path_fig.show(config={"toImageButtonOptions": {"format": "svg"}})


if save:
    V2D_fig.write_image(f"{sim_image_path}/V2D_{idx}.svg", format='svg', scale=3)
    V2D_path_fig.write_image(f"{sim_image_path}/V2D_path_{idx}.svg", format='svg', scale=3)


In [ ]:
root_name = dend_name
target_dend = 'dend[15]'
extracted = extract_dend_to_target(dend_tree, root_name, target_dend)
extracted

# Rebuild idxs from the new extracted structure
idxs = []
for outer_list in extracted:
    outer_idxs = []
    for path in outer_list:
        path_idxs = []
        for dend in path:
            dendname = dend.name()
            idx = get_coord_index(cell_coordinates, dendname, None)
            path_idxs.extend(idx)
        outer_idxs.append(path_idxs)
    idxs.append(outer_idxs)

# Rebuild dists
dists = []
for outer_list in idxs:
    outer_dists = []
    for path_idxs in outer_list:
        path_dists = cell_coordinates[path_idxs, 5].astype(float)
        outer_dists.append(path_dists)
    dists.append(outer_dists)


# Build Vpeaks (use 0 for first simulation, not idx)
sim_idx = 0

Vpeaks = [[Vpeaks_3D[sim_idx][np.array(path_idxs)] for path_idxs in outer_list] for outer_list in idxs]

titles = [path[-1].name() for outer_list in extracted for path in outer_list]

fig = plot_v_mpl(dists[0], Vpeaks[0], titles=titles, colors='darkgray', xrange=[-5, 275], yrange=[-85, 40], height=500, width=500,  points_size=4)

fig

if save:
    fig.savefig(f"{sim_image_path}/voltage_vs_distance_{sim_idx}.svg", format='svg', dpi=300, bbox_inches='tight')



In [ ]:
summarise_sim_entry(sim_data, 'i_mechs_3D')
    

In [ ]:
mechs2display = get_mechs2display(sim_data)
name_map = {
    'kaf': 'K<sub>v</sub>4',
    'kas': 'K<sub>v</sub>1',
    'kir': 'K<sub>ir</sub>',
    'kcnq': 'K<sub>v</sub>7'
}


idx1 = get_coord_index(cell_coordinates, 'soma[0]', 0.5)
idx2 = get_coord_index(cell_coordinates, record_dendrite, record_location)

for mech_name in mechs2display:
    I_all = extract_mech_currents(sim_data, mech_name, [idx1, idx2])
    soma_traces = I_all[idx1]
    dend_traces = I_all[idx2]

    # convert to µA/cm²
    soma_traces = [trace * 1e3 for trace in soma_traces]
    dend_traces = [trace * 1e3 for trace in dend_traces]

    # compute yrange across both soma & dend
    yrange_soma = get_y_range(soma_traces)
    yrange_dend = get_y_range(dend_traces)

    span_soma = yrange_soma[1] - yrange_soma[0]
    y_err_bar_soma = span_soma * 0.10
    y_err_bar_soma = roundup(y_err_bar_soma)
    
    span_dend = yrange_dend[1] - yrange_dend[0]
    y_err_bar_dend = span_dend * 0.10
    y_err_bar_dend = roundup(y_err_bar_dend)

    mech_label = name_map.get(mech_name, f'K<sub>{mech_name}</sub>')

    soma_fig = plot3_dt(soma_traces, yaxis='', yrange=yrange_soma, stim_time=150, sim_time=sim_time, height=height2, width=width2,
                        title=f'somatic {mech_label} density (µA/cm²)', y_err_bar_units='µA/cm²',
                        black_trace=None, gray_trace=None, y_err_bar=y_err_bar_soma, 
                        x_err_bar=50, baseline=50, dt=dt, ds=ds_plot, offset=True, offset_ms=150, offset_y=None)

    dend_fig = plot3_dt(dend_traces, yaxis='', yrange=yrange_dend, stim_time=150, sim_time=sim_time, height=height2, width=width2, 
                        title=f'dendritic {mech_label} density (µA/cm²)', y_err_bar_units='µA/cm²',
                            black_trace=None, gray_trace=None, y_err_bar=y_err_bar_soma, 
                        x_err_bar=50, baseline=50, dt=dt, ds=ds_plot, offset=True, offset_ms=150, offset_y=None)

    soma_fig.show(config={"toImageButtonOptions": {"format": "svg"}})
    dend_fig.show(config={"toImageButtonOptions": {"format": "svg"}})

    if save:
        soma_fig.write_image(f"{sim_image_path}/current_density_soma_{mech_name}.svg", format='svg', scale=3)
        dend_fig.write_image(f"{sim_image_path}/current_density_dend_{mech_name}.svg", format='svg', scale=3)



In [ ]:
n_gabas = [sim_data[sim_id]['metadata']['num_gabas'] for sim_id in sim_data.keys()]


n_spikes = []
for i, trace in enumerate(Vsoma):
    n_spikes.append(count_spikes(trace[0], dt=dt))

df = pd.DataFrame({
    'n_gabas': n_gabas,
    'n_spikes': n_spikes
})

plot_df(df, x_col='n_gabas', y_cols=['n_spikes'], xrange=[-1, 50], yrange=[-0.08125,3.25], xtick_interval=10, ytick_interval=1,
        sim_image_path=sim_image_path, save=save, dashed_lines=False, marker_size=3, open_circles=False, plot_color='#6961ab',
        y_title=r'$N_{spikes}$', x_title = r'$N_{GABA_R}$',  title='', width=2.5, height=1.3, save_name='spikes_vs_gabas.svg', legend=False)